# SHROOM RAG E5 — Module 2 Holdout Evaluation

Μοναδική εγκεκριμένη offline unblinded αξιολόγηση του παγωμένου Module 2 holdout. Η configuration, το threshold, οι ανθρώπινες annotations και οι gate decisions δεν αλλάζουν.

Η `Q_SRC_TGT` είναι reference-assisted. Τα 10 holdout examples αποτελούν μικρό προκαθορισμένο diagnostic holdout και όχι external test set. Δεν εκτελείται retrieval, dense encoding, Gemini/API ή official test.


In [ ]:
# 1. Setup: μόνο τα πέντε εγκεκριμένα evaluation inputs
from __future__ import annotations
import csv, hashlib, io, json, os, pathlib, statistics, tempfile
from collections import Counter


# Προσαρμοσμένος κώδικας και ιστορικά τεκμήρια έχουν χωριστές ταυτότητες.
import importlib.util as _module2_import
import sys as _module2_sys
from pathlib import Path as _Module2Path
_module2_sys.dont_write_bytecode = True
_module2_code_root = _Module2Path(os.environ["SHROOM_MODULE2_CODE_ROOT"])
if not _module2_code_root.is_absolute() or ".." in _module2_code_root.parts:
    raise ValueError("SHROOM_MODULE2_CODE_ROOT must be an absolute local directory.")
_module2_helper_path = _module2_code_root / "scripts" / "module2_dense_cache.py"
for _part in (_module2_helper_path, *_module2_helper_path.parents):
    if _part.is_symlink() or (getattr(_part.lstat(), "st_file_attributes", 0) & 0x400):
        raise ValueError("Reparse/symlink helper paths are not accepted.")
_module2_raw = _module2_helper_path.read_bytes()
if hashlib.sha256(_module2_raw).hexdigest() != os.environ["SHROOM_MODULE2_HELPER_SHA256"].lower():
    raise ValueError("Adapted helper pin mismatch; no fallback to historical source.")
if "module2_dense_cache" in _module2_sys.modules:
    _module2 = _module2_sys.modules["module2_dense_cache"]
    if _Module2Path(_module2.__file__) != _module2_helper_path or getattr(_module2, "_notebook_pin", None) != hashlib.sha256(_module2_raw).hexdigest():
        raise ValueError("A different/stale helper is already loaded; restart the kernel.")
else:
    _spec = _module2_import.spec_from_file_location("module2_dense_cache", _module2_helper_path)
    _module2 = _module2_import.module_from_spec(_spec)
    _module2_sys.modules["module2_dense_cache"] = _module2
    _spec.loader.exec_module(_module2)
    _module2._notebook_pin = hashlib.sha256(_module2_raw).hexdigest()
CTX = _module2.context_from_environment(require_cache=False)
print("Adapted Module 2: H0 evidence guards retained; runtime reproduction NOT VERIFIED.")

ROOT = CTX.evidence("RAG")
ANNOTATION = ROOT / "results" / "rag_module2" / "annotation"
HOLDOUT = ROOT / "results" / "rag_module2" / "holdout"
EVALUATION = ROOT / "results" / "rag_module2" / "evaluation"
INPUTS = {
    "blinded": ANNOTATION / "module2_annotation_holdout_blinded.csv",
    "human": ANNOTATION / "module2_annotation_holdout_human_completed.csv",
    "manifest": HOLDOUT / "module2_holdout_frozen_manifest.json",
    "config": EVALUATION / "module2_selected_retrieval_config.json",
    "preparation_report": HOLDOUT / "MODULE2_HOLDOUT_PREPARATION.md",
}
OUTPUTS = {
    "metrics": CTX.run_dir / "holdout" / "module2_holdout_metrics.json",
    "decisions": CTX.run_dir / "holdout" / "module2_holdout_decisions_unblinded.csv",
    "report": CTX.run_dir / "holdout" / "MODULE2_HOLDOUT_RESULTS.md",
}
EXPECTED_HASHES = {
    "blinded": "283be261ddfe126b28016f7155db75f8742ca04d9e17530727413f84cb9b03c2",
    "human": "7519ae4988cf2bd5d0678d83523b884936beb6b257b0bff249f0de893531a93e",
    "manifest": "f7356c17dbf04e37f6125c13e1484a174490c3e8598596ba253010b846106c22",
    "config": "f9e8b35417222a97a29c17cd4d27b3140c9658f1c1ef84068eb61b72d63f572c",
    "preparation_report": "6399bda978a48b27cf6de04773858406c5d4b41df22cc69d56dc7d8dc1eab0d6",
}
REQUIRED_COLUMNS = ["annotation_id","example_id","term","src","tgt","evidence_id","candidate_term","candidate_definition","candidate_context","relevance_category","annotation_confidence","annotation_notes"]
STRICT = {"relevant_same_sense"}
LENIENT = {"relevant_same_sense", "partially_relevant"}
UNSAFE = {"different_sense", "irrelevant"}
ALL_CATEGORIES = ["relevant_same_sense","partially_relevant","different_sense","irrelevant","uncertain_review"]
ALLOWED_CONFIDENCES = {"high","medium","low"}
FROZEN_THRESHOLD = 42.555018684798945
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


## 2. Preflight πριν από metrics

Κάθε assertion εκτελείται πριν από την παραγωγή metrics ή outputs. Το ανθρώπινο CSV πρέπει να είναι ακριβές συμπλήρωμα του blinded CSV και το manifest να δίνει ακριβώς 2 accepted και 8 abstained frozen decisions.


In [ ]:
# 2. Read-only integrity helpers και locked snapshots
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest().lower()

def read_csv(path):
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        return list(reader), list(reader.fieldnames or [])

for name, path in INPUTS.items():
    assert path.exists(), (name, path)
    assert sha256(path) == EXPECTED_HASHES[name], (name, sha256(path))

blinded_rows, blinded_fields = read_csv(INPUTS["blinded"])
human_rows, human_fields = read_csv(INPUTS["human"])
manifest = json.loads(INPUTS["manifest"].read_text(encoding="utf-8"))
config_payload = json.loads(INPUTS["config"].read_text(encoding="utf-8"))

# Αυτά τα artifacts χρησιμοποιούνται μόνο ως before/after hash guards, ποτέ ως evaluation data.
PROTECTED = [
    ROOT / "shroom_ensemble.ipynb",
    ROOT / "shroom_rag_e5_module1.ipynb",
    ROOT / "shroom_rag_e5_module1_gemini_trial.ipynb",
    ROOT / "shroom_rag_e5_module2_retrieval.ipynb",
    ROOT / "shroom_rag_e5_module2_phase_b_evaluation.ipynb",
    ROOT / "scripts" / "module2_dense_cache.py",
    ROOT / "results" / "rag_module2" / "module2_candidates_top5.jsonl",
    ROOT / "results" / "rag_module2" / "module2_input_manifest.json",
    ROOT / "results" / "rag_module2" / "MODULE2_PHASE_A_RESULTS.md",
    EVALUATION / "module2_dev_retrieval_metrics.csv",
    EVALUATION / "module2_dev_gate_sensitivity.csv",
    EVALUATION / "module2_dev_decisions.jsonl",
    EVALUATION / "MODULE2_PHASE_B_RESULTS.md",
    ANNOTATION / "module2_annotation_dev_blinded.csv",
    ANNOTATION / "module2_annotation_dev_human_completed.csv",
    ANNOTATION / "MODULE2_ANNOTATION_GUIDE.md",
] + list(INPUTS.values())
protected_before = {str(path): sha256(path) for path in PROTECTED}


In [ ]:
# 3. Mandatory preflight assertions
assert len(human_rows) == len(blinded_rows) == 10
assert human_fields == blinded_fields == REQUIRED_COLUMNS
assert [r["annotation_id"] for r in human_rows] == manifest["ordered_annotation_ids"] == [r["annotation_id"] for r in blinded_rows]
assert len({r["annotation_id"] for r in human_rows}) == 10
assert len({r["example_id"] for r in human_rows}) == 10
assert len({r["evidence_id"] for r in human_rows}) == 10
assert all(r["relevance_category"] and r["annotation_confidence"] for r in human_rows)
assert {r["relevance_category"] for r in human_rows} <= set(ALL_CATEGORIES)
assert {r["annotation_confidence"] for r in human_rows} <= ALLOWED_CONFIDENCES
assert all([h[k] for k in REQUIRED_COLUMNS[:9]] == [b[k] for k in REQUIRED_COLUMNS[:9]] for h,b in zip(human_rows,blinded_rows))
forbidden = {"method","retrieval_method","rank","method_rank","score","bm25_score","method_agreement","gate_decision","frozen_gate_decision","label","labels","label_bin","p(Hallucination)","p_hallucination","probability"}
assert not (forbidden & set(human_fields))

frozen = config_payload["selected_configuration"]
assert frozen["query_variant"] == "Q_SRC_TGT"
assert frozen["retrieval_method"] == "bm25"
assert frozen["score_threshold"] == FROZEN_THRESHOLD
assert frozen["min_method_agreement"] == 2
assert frozen["reject_duplicates"] is True
assert frozen["relative_margin_threshold"] is None
assert frozen["require_exact_match"] is False
assert manifest["frozen_input_hashes"]["selected_retrieval_config_sha256"] == EXPECTED_HASHES["config"]
assert manifest["blinded_csv"]["sha256"] == EXPECTED_HASHES["blinded"]

manifest_by_id = {r["annotation_id"]: r for r in manifest["holdout_examples"]}
assert len(manifest_by_id) == 10
assert list(manifest_by_id) == manifest["ordered_annotation_ids"]
frozen_decisions = [manifest_by_id[r["annotation_id"]]["frozen_gate_decision"] for r in human_rows]
assert frozen_decisions.count("accept") == 2
assert frozen_decisions.count("abstain") == 8
assert len({manifest_by_id[r["annotation_id"]]["evidence_id"] for r in human_rows}) == 10
assert all(str(manifest_by_id[r["annotation_id"]]["evidence_id"]) == r["evidence_id"] for r in human_rows)
print("PREFLIGHT PASS — 10/10 join; frozen decisions 2 accepted / 8 abstained")


## 3. Metric definitions και unit tests

Όλα τα rates διατηρούν count και denominator. Το accepted usable coverage χρησιμοποιεί παρονομαστή 10. Η precision/proportion μεταξύ accepted χρησιμοποιεί παρονομαστή 2. Τα false-abstention rates παρουσιάζονται τόσο στο σύνολο `/10` όσο και μεταξύ abstained `/8`.


In [ ]:
# 4. Deterministic metric helpers και synthetic unit tests
def ratio(count, denominator):
    assert isinstance(count, int) and isinstance(denominator, int)
    assert 0 <= count <= denominator and denominator > 0
    return {"count": count, "denominator": denominator, "rate": count / denominator}

def distribution(labels, categories, denominator):
    counts = Counter(labels)
    return {category: ratio(counts.get(category, 0), denominator) for category in categories}

unit_tests = []
def test(name, function):
    try:
        function(); unit_tests.append({"name":name,"passed":True})
    except Exception as exc:
        unit_tests.append({"name":name,"passed":False,"error":str(exc)})

def assert_true(value): assert value
test("ratio count/denominator", lambda: assert_true(ratio(2,10) == {"count":2,"denominator":10,"rate":0.2}))
test("strict synthetic count", lambda: assert_true(sum(x in STRICT for x in ["relevant_same_sense","irrelevant"]) == 1))
test("lenient synthetic count", lambda: assert_true(sum(x in LENIENT for x in ["relevant_same_sense","partially_relevant","irrelevant"]) == 2))
test("unsafe synthetic count", lambda: assert_true(sum(x in UNSAFE for x in ["different_sense","irrelevant","uncertain_review"]) == 2))
test("distribution denominator", lambda: assert_true(distribution(["irrelevant","irrelevant"],ALL_CATEGORIES,2)["irrelevant"]["rate"] == 1.0))
assert all(x["passed"] for x in unit_tests), unit_tests
print(f"METRIC UNIT TESTS PASS — {len(unit_tests)}/{len(unit_tests)}")


In [ ]:
# 5. Immutable 10/10 join και unblinded decision table
decision_rows = []
for order, annotation in enumerate(human_rows, 1):
    frozen_row = manifest_by_id[annotation["annotation_id"]]
    category = annotation["relevance_category"]
    decision_rows.append({
        "order": order,
        **{key: annotation[key] for key in REQUIRED_COLUMNS},
        "query_variant": "Q_SRC_TGT",
        "retrieval_method": "bm25",
        "bm25_score": frozen_row["bm25_score"],
        "method_agreement": frozen_row["method_agreement"],
        "duplicate_detected": frozen_row["duplicate_detected"],
        "frozen_gate_decision": frozen_row["frozen_gate_decision"],
        "abstention_reasons": "|".join(frozen_row["abstention_reasons"]),
        "strict_relevant": category in STRICT,
        "lenient_relevant": category in LENIENT,
        "unsafe": category in UNSAFE,
    })
assert len(decision_rows) == 10
assert len({r["annotation_id"] for r in decision_rows}) == 10
assert len({r["example_id"] for r in decision_rows}) == 10
assert len({r["evidence_id"] for r in decision_rows}) == 10
assert sum(r["frozen_gate_decision"] == "accept" for r in decision_rows) == 2
assert sum(r["frozen_gate_decision"] == "abstain" for r in decision_rows) == 8


In [ ]:
# 6. Exact requested metrics with explicit counts and denominators
n_total = 10
accepted = [r for r in decision_rows if r["frozen_gate_decision"] == "accept"]
abstained = [r for r in decision_rows if r["frozen_gate_decision"] == "abstain"]
assert len(accepted) == 2 and len(abstained) == 8

all_labels = [r["relevance_category"] for r in decision_rows]
accepted_labels = [r["relevance_category"] for r in accepted]
abstained_labels = [r["relevance_category"] for r in abstained]

ungated = {
    "strict_relevant_at_1": ratio(sum(x in STRICT for x in all_labels), 10),
    "lenient_relevant_at_1": ratio(sum(x in LENIENT for x in all_labels), 10),
    "partially_relevant_rate": ratio(sum(x == "partially_relevant" for x in all_labels), 10),
    "different_sense_rate": ratio(sum(x == "different_sense" for x in all_labels), 10),
    "irrelevant_rate": ratio(sum(x == "irrelevant" for x in all_labels), 10),
    "unsafe_top1_rate": ratio(sum(x in UNSAFE for x in all_labels), 10),
    "uncertain_review_rate": ratio(sum(x == "uncertain_review" for x in all_labels), 10),
}

strict_accepted = sum(x in STRICT for x in accepted_labels)
lenient_accepted = sum(x in LENIENT for x in accepted_labels)
unsafe_accepted = sum(x in UNSAFE for x in accepted_labels)
strict_false_abstentions = sum(x in STRICT for x in abstained_labels)
lenient_false_abstentions = sum(x in LENIENT for x in abstained_labels)

gated = {
    "accepted_coverage": ratio(len(accepted), 10),
    "abstention_rate": ratio(len(abstained), 10),
    "strict_accepted_usable_coverage": ratio(strict_accepted, 10),
    "lenient_accepted_usable_coverage": ratio(lenient_accepted, 10),
    "unsafe_accepted_evidence_rate_total": ratio(unsafe_accepted, 10),
    "strict_precision_among_accepted": ratio(strict_accepted, len(accepted)),
    "lenient_precision_among_accepted": ratio(lenient_accepted, len(accepted)),
    "unsafe_proportion_among_accepted": ratio(unsafe_accepted, len(accepted)),
    "strict_false_abstentions": {
        "count": strict_false_abstentions,
        "rate_total": ratio(strict_false_abstentions, 10),
        "rate_among_abstained": ratio(strict_false_abstentions, len(abstained)),
    },
    "lenient_false_abstentions": {
        "count": lenient_false_abstentions,
        "rate_total": ratio(lenient_false_abstentions, 10),
        "rate_among_abstained": ratio(lenient_false_abstentions, len(abstained)),
    },
    "accepted_relevance_distribution": distribution(accepted_labels, ALL_CATEGORIES, len(accepted)),
    "abstained_relevance_distribution": distribution(abstained_labels, ALL_CATEGORIES, len(abstained)),
}
print("UNGATED", json.dumps(ungated, indent=2))
print("GATED", json.dumps(gated, indent=2))


In [ ]:
# 7. Predeclared criteria, without retuning
criteria = {
    "accepted_unsafe_evidence_zero": {
        "actual": gated["unsafe_proportion_among_accepted"],
        "required_rate_maximum": 0.0,
        "passed": gated["unsafe_proportion_among_accepted"]["rate"] == 0.0,
    },
    "strict_accepted_usable_coverage_at_least_20_percent": {
        "actual": gated["strict_accepted_usable_coverage"],
        "required_rate_minimum": 0.20,
        "passed": gated["strict_accepted_usable_coverage"]["rate"] >= 0.20,
    },
    "lenient_accepted_usable_coverage_at_least_30_percent": {
        "actual": gated["lenient_accepted_usable_coverage"],
        "required_rate_minimum": 0.30,
        "passed": gated["lenient_accepted_usable_coverage"]["rate"] >= 0.30,
    },
    "not_abstain_all": {
        "actual_accepted": ratio(len(accepted), 10),
        "passed": len(accepted) > 0,
    },
    "configuration_unchanged": {
        "selected_config_sha256": sha256(INPUTS["config"]),
        "passed": sha256(INPUTS["config"]) == EXPECTED_HASHES["config"],
    },
}
holdout_criteria_pass = all(item["passed"] for item in criteria.values())
development_regression_blocker = {
    "failed_strict_cases": ["preciosity","weary","inconsistent"],
    "failed_count": 3,
    "gemini_blocked_independently": True,
}
final_decision = "GO" if holdout_criteria_pass and development_regression_blocker["failed_count"] == 0 else "NO_GO_FOR_GEMINI"
assert final_decision == "NO_GO_FOR_GEMINI"
print("FINAL DECISION:", final_decision)


## 4. Deterministic validation και atomic outputs

Τα outputs γράφονται atomically μόνο αφού περάσουν preflight, metric tests, join tests και frozen-decision checks. Δεν αντικαθίσταται κανένα blinded ή human-completed αρχείο.


In [ ]:
# 8. Full deterministic test suite
tests = list(unit_tests)
def add_test(name, condition):
    tests.append({"name":name,"passed":bool(condition)})
    assert condition, name

add_test("input hashes frozen", all(sha256(INPUTS[k]) == v for k,v in EXPECTED_HASHES.items()))
add_test("10/10 full join", len(decision_rows) == 10)
add_test("no missing IDs", all(r["annotation_id"] and r["example_id"] and r["evidence_id"] for r in decision_rows))
add_test("no duplicate annotation IDs", len({r["annotation_id"] for r in decision_rows}) == 10)
add_test("no duplicate example IDs", len({r["example_id"] for r in decision_rows}) == 10)
add_test("no duplicate evidence IDs", len({r["evidence_id"] for r in decision_rows}) == 10)
add_test("frozen decisions unchanged", [r["frozen_gate_decision"] for r in decision_rows] == frozen_decisions)
add_test("accepted/abstained fixed 2/8", len(accepted) == 2 and len(abstained) == 8)
add_test("frozen configuration exact", frozen["score_threshold"] == FROZEN_THRESHOLD and frozen["min_method_agreement"] == 2 and frozen["reject_duplicates"] is True)
add_test("annotations complete", all(r["relevance_category"] and r["annotation_confidence"] for r in decision_rows))
add_test("decision deterministic", final_decision == "NO_GO_FOR_GEMINI")
print(f"TESTS PASS — {len(tests)}/{len(tests)}")


In [ ]:
# 9. Atomic JSON, CSV και Markdown writers
def atomic_text(path, text):
    CTX.write_text(path, text)

metrics_payload = {
    "phase": "SHROOM RAG E5 Module 2 frozen holdout evaluation",
    "scientific_scope": "Small predeclared reference-assisted holdout; not an external test set.",
    "input_hashes": {name:sha256(path) for name,path in INPUTS.items()},
    "frozen_configuration": {
        "query_variant":"Q_SRC_TGT",
        "reference_assisted":True,
        "retriever":"bm25",
        "bm25_score_minimum":FROZEN_THRESHOLD,
        "minimum_method_agreement":2,
        "duplicate_rejection":True,
        "retuning_or_comparison_performed":False,
    },
    "sample_size":10,
    "frozen_decision_counts":{"accepted":2,"abstained":8},
    "ungated_frozen_retriever_top1":ungated,
    "frozen_gate":gated,
    "predeclared_criteria":criteria,
    "holdout_criteria_pass":holdout_criteria_pass,
    "development_regression_blocker":development_regression_blocker,
    "final_decision":final_decision,
    "tests":{"passed":sum(x["passed"] for x in tests),"total":len(tests),"details":tests},
    "operations":{"retrieval_or_dense_runs":0,"downloads_or_installations":0,"official_test":0,"gemini_api_or_network":0,"configuration_changes":0,"annotation_changes":0},
}
atomic_text(OUTPUTS["metrics"], json.dumps(metrics_payload, ensure_ascii=False, indent=2, sort_keys=True)+"\n")

decision_fields = list(decision_rows[0])
buffer = io.StringIO(newline="")
writer = csv.DictWriter(buffer, fieldnames=decision_fields, lineterminator="\n")
writer.writeheader(); writer.writerows(decision_rows)
atomic_text(OUTPUTS["decisions"], buffer.getvalue())

def display_metric(metric):
    return f"{metric['count']}/{metric['denominator']} ({metric['rate']*100:.1f}%)"

lines = [
    "# SHROOM RAG E5 — Module 2 Holdout Results","",
    "## Integrity and scope","",
    "- Preflight: PASS — 10 rows × 12 columns, complete ordered 10/10 join, unique example/evidence IDs.",
    f"- Human-completed SHA-256: `{sha256(INPUTS['human'])}`.",
    f"- Blinded SHA-256: `{sha256(INPUTS['blinded'])}`.",
    f"- Frozen manifest SHA-256: `{sha256(INPUTS['manifest'])}`.",
    f"- Selected configuration SHA-256: `{sha256(INPUTS['config'])}`.",
    f"- Deterministic tests: {len(tests)}/{len(tests)} passed.",
    "- Frozen configuration: reference-assisted Q_SRC_TGT + BM25; score >= 42.555018684798945; method agreement >= 2; duplicate rejection.",
    "- No retuning or configuration comparison was performed.","",
    "## Ungated frozen-retriever top-1","",
    f"- Strict Relevant@1: {display_metric(ungated['strict_relevant_at_1'])}",
    f"- Lenient Relevant@1: {display_metric(ungated['lenient_relevant_at_1'])}",
    f"- Partially relevant: {display_metric(ungated['partially_relevant_rate'])}",
    f"- Different sense: {display_metric(ungated['different_sense_rate'])}",
    f"- Irrelevant: {display_metric(ungated['irrelevant_rate'])}",
    f"- Unsafe top-1: {display_metric(ungated['unsafe_top1_rate'])}","",
    "## Frozen gate","",
    f"- Accepted coverage: {display_metric(gated['accepted_coverage'])}",
    f"- Abstention rate: {display_metric(gated['abstention_rate'])}",
    f"- Strict accepted usable coverage: {display_metric(gated['strict_accepted_usable_coverage'])}",
    f"- Lenient accepted usable coverage: {display_metric(gated['lenient_accepted_usable_coverage'])}",
    f"- Unsafe accepted evidence / all holdout: {display_metric(gated['unsafe_accepted_evidence_rate_total'])}",
    f"- Strict precision among accepted: {display_metric(gated['strict_precision_among_accepted'])}",
    f"- Lenient precision among accepted: {display_metric(gated['lenient_precision_among_accepted'])}",
    f"- Unsafe proportion among accepted: {display_metric(gated['unsafe_proportion_among_accepted'])}",
    f"- Strict false abstentions: {gated['strict_false_abstentions']['count']}; {display_metric(gated['strict_false_abstentions']['rate_total'])} overall; {display_metric(gated['strict_false_abstentions']['rate_among_abstained'])} among abstained.",
    f"- Lenient false abstentions: {gated['lenient_false_abstentions']['count']}; {display_metric(gated['lenient_false_abstentions']['rate_total'])} overall; {display_metric(gated['lenient_false_abstentions']['rate_among_abstained'])} among abstained.","",
    "### Accepted relevance breakdown (denominator 2)","",
]
for category in ALL_CATEGORIES: lines.append(f"- `{category}`: {display_metric(gated['accepted_relevance_distribution'][category])}")
lines += ["","### Abstained relevance breakdown (denominator 8)",""]
for category in ALL_CATEGORIES: lines.append(f"- `{category}`: {display_metric(gated['abstained_relevance_distribution'][category])}")
lines += ["","## Predeclared decision criteria","",
    f"- Accepted unsafe evidence = 0%: **{'PASS' if criteria['accepted_unsafe_evidence_zero']['passed'] else 'FAIL'}** — {display_metric(gated['unsafe_proportion_among_accepted'])}.",
    f"- Strict accepted usable coverage >= 20%: **{'PASS' if criteria['strict_accepted_usable_coverage_at_least_20_percent']['passed'] else 'FAIL'}** — {display_metric(gated['strict_accepted_usable_coverage'])}.",
    f"- Lenient accepted usable coverage >= 30%: **{'PASS' if criteria['lenient_accepted_usable_coverage_at_least_30_percent']['passed'] else 'FAIL'}** — {display_metric(gated['lenient_accepted_usable_coverage'])}.",
    f"- Abstain-all prohibited: **{'PASS' if criteria['not_abstain_all']['passed'] else 'FAIL'}** — {display_metric(gated['accepted_coverage'])} accepted.",
    f"- Configuration unchanged: **{'PASS' if criteria['configuration_unchanged']['passed'] else 'FAIL'}**.","",
    f"## Final decision: `{final_decision}`","",
    "The safety criterion passes, but strict and lenient accepted usable coverage fail their predeclared thresholds. Thresholds were not relaxed and no alternative configuration was examined.",
    "Gemini is independently blocked because the strict development regression cases `preciosity`, `weary`, and `inconsistent` remain failed.","",
    "## Scientific interpretation","",
    "This is a very small, reference-assisted holdout. It supports only a diagnostic statement about this frozen configuration and cannot be interpreted as independent external-test evidence or a generalization estimate.","",
    "Official test: 0; retrieval/dense runs: 0; Gemini/API/network: 0; downloads/installations: 0; annotation/configuration changes: 0.","",
]
atomic_text(OUTPUTS["report"], "\n".join(lines))
print("Atomic outputs written:", len(OUTPUTS))


In [ ]:
# 10. Final validation after writes
for path in OUTPUTS.values(): assert path.exists() and path.stat().st_size > 0
written_metrics = json.loads(OUTPUTS["metrics"].read_text(encoding="utf-8"))
written_decisions, written_fields = read_csv(OUTPUTS["decisions"])
assert written_metrics["final_decision"] == "NO_GO_FOR_GEMINI"
assert written_metrics["frozen_decision_counts"] == {"accepted":2,"abstained":8}
assert len(written_decisions) == 10 and len({r["annotation_id"] for r in written_decisions}) == 10
assert {str(path):sha256(path) for path in PROTECTED} == protected_before
assert all(sha256(INPUTS[name]) == expected for name,expected in EXPECTED_HASHES.items())
artifact_hashes = {path.name:sha256(path) for path in OUTPUTS.values()}
print("FINAL VALIDATION PASS")
print(json.dumps(artifact_hashes, indent=2, sort_keys=True))


## Stop condition

Η holdout αξιολόγηση σταματά εδώ. Δεν ξεκινά Gemini, Module 3, νέο gate design ή E3 integration.
